# Κεφάλαιο 13: Προσαρμογή του μοντέλου σου με περαιτέρω εκπαίδευση (fine-tuning)

> «Τύχη είναι αυτό που συμβαίνει όταν η προετοιμασία συναντά την ευκαιρία».
> — **Σενέκας**

---

## Τι θα μάθεις

- Πότε να προσαρμόσεις το μοντέλο με περαιτέρω εκπαίδευση και πότε να χρησιμοποιήσεις καλύτερες προτροπές
- Πώς να ετοιμάσεις δεδομένα εκπαίδευσης με ζεύγη οδηγίας/απάντησης
- Γιατί η μάσκα απώλειας βοηθά την εκπαίδευση να εστιάσει σε ό,τι έχει σημασία
- Πώς ένας προσαρμογέας LoRA βαθμού 8 χρησιμοποιεί μόλις το 1/48 των εκπαιδεύσιμων βαρών για μια προβολή 768×768
- Πώς να αξιολογήσεις τη συμπεριφορά του μοντέλου πριν και μετά την προσαρμογή

---

## Ρύθμιση περιβάλλοντος

Άνοιξε το notebook [απευθείας στο Colab](https://colab.research.google.com/github/FirstLLM/code/blob/main/notebooks/el/ch13.ipynb) ή κατέβασε το [αρχείο για το Jupyter](https://github.com/FirstLLM/code/blob/main/notebooks/el/ch13.ipynb).

Πρώτα, ας εγκαταστήσουμε τα απαραίτητα πακέτα και ας ελέγξουμε αν υπάρχει διαθέσιμη GPU.

In [ ]:
# Εγκατάστησε τα απαραίτητα πακέτα (οι εκδόσεις είναι αυτές που δοκιμάστηκαν για το βιβλίο,
# βλ. Παράρτημα A· τα torch/transformers εισάγονται απευθείας στο επόμενο κελί)
!pip install -q torch==2.15.0 transformers==5.23.0 tqdm==4.70.1 numpy==2.3.5 matplotlib==3.10.7

In [ ]:
# ===== ΕΙΣΑΓΩΓΕΣ =====
import math
import json
import os
from dataclasses import dataclass
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer
from tqdm import tqdm

# Έλεγξε αν υπάρχει GPU
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Συσκευή σε χρήση: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Μνήμη: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif device.type == "mps":
    print("Χρησιμοποιείται η GPU μέσω MPS.")
else:
    print("ΠΡΟΕΙΔΟΠΟΙΗΣΗ: Δεν υπάρχει διαθέσιμη GPU για την εκπαίδευση· θα χρησιμοποιηθεί η CPU.")
    print("Αν χρησιμοποιείς Colab, μπορείς να επιλέξεις Runtime > Change runtime type > GPU.")

In [ ]:
# ===== ΑΝΑΠΑΡΑΓΩΓΙΜΟΤΗΤΑ =====
import random
import numpy as np

def set_seed(seed=42):
    """Όρισε την τιμή seed για κάθε πηγή τυχαιότητας που χρησιμοποιούμε (ίδια βοηθητική συνάρτηση όπως στο Κεφάλαιο 12)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)


## 1. Τα μέρη του μοντέλου από τα Κεφάλαια 10-12

Πρώτα, ας εισαγάγουμε το μοντέλο MiniGPT που φτιάξαμε στα προηγούμενα κεφάλαια.

In [ ]:
# ===== ΠΡΟΣΟΧΗ ΠΟΛΛΑΠΛΩΝ ΚΕΦΑΛΩΝ (από το Κεφάλαιο 10) =====
class MultiHeadAttention(nn.Module):
    """
    Αποδοτική προσοχή πολλαπλών κεφαλών (επεξεργάζεται όλες τις κεφαλές μαζί σε μία παρτίδα).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "Το d_model πρέπει να διαιρείται ακριβώς με το num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64

        # Μία ενιαία προβολή υπολογίζει τα Q, K και V αντί για τρεις ξεχωριστές προβολές.
        # Γιατί 3 * d_model; Επειδή προβάλλουμε ταυτόχρονα στα Q, K, V
        # Η ρύθμιση bias=True μάς επιτρέπει να φορτώσουμε και τους προεκπαιδευμένους όρους μετατόπισης του GPT-2.
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # Προβολή εξόδου
        self.out_proj = nn.Linear(d_model, d_model)

        # Εφαρμογή dropout στα βάρη προσοχής
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        # ===== Βήμα 1: Κάνε την προβολή στα Q, K, V (όλα μαζί!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)

        # ===== Βήμα 2: Χώρισε το αποτέλεσμα σε Q, K, V και προσάρμοσε το σχήμα για τις πολλαπλές κεφαλές =====
        # Άλλαξε το σχήμα σε (batch, seq, 3, num_heads, d_head)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)

        # Άλλαξε τη σειρά των διαστάσεων σε (3, batch, num_heads, seq, d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Χώρισε τον τανυστή σε Q, K, V: το καθένα έχει σχήμα (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # ===== Βήμα 3: Προσοχή με κλιμακωτό εσωτερικό γινόμενο (όλες οι κεφαλές σε μία παρτίδα) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)

        # Εφάρμοσε την αιτιακή μάσκα, αν έχει δοθεί
        if mask is not None:
            # Επέκτεινε τη μάσκα για τις κεφαλές: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)

        # Σταθμισμένο άθροισμα των τιμών
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)

        # ===== Βήμα 4: Συνένωσε τις κεφαλές =====
        # Άλλαξε τη σειρά των αξόνων σε (batch, seq, num_heads, d_head)
        attn_output = attn_output.transpose(1, 2)

        # Άλλαξε το σχήμα σε (batch, seq, d_model) για να συνενώσεις τις κεφαλές
        attn_output = attn_output.reshape(batch, seq, d_model)

        # ===== Βήμα 5: Τελική προβολή =====
        output = self.out_proj(attn_output)

        return output, attn_weights


# Δοκίμασέ το

print("Ορίστηκε η κλάση MultiHeadAttention!")

In [ ]:
# ===== ΔΙΚΤΥΟ ΠΡΟΣΘΙΑΣ ΤΡΟΦΟΔΟΤΗΣΗΣ (από το Κεφάλαιο 10) =====
class FeedForward(nn.Module):
    """
    Δίκτυο πρόσθιας τροφοδότησης ανά θέση.
    Εφαρμόζεται σε κάθε θέση ξεχωριστά (με τα ίδια βάρη για όλες τις θέσεις).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Ορίσματα:
            d_model: Διάσταση του μοντέλου (768 για το GPT-2 small)
            d_ff: Ενδιάμεση διάσταση του δικτύου πρόσθιας τροφοδότησης (συνήθως 4 * d_model = 3072)
            dropout: Πιθανότητα απενεργοποίησης (dropout)
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff), αύξησε την τελευταία διάσταση
        x = F.gelu(x, approximate="tanh")  # Μη γραμμική συνάρτηση (η παραλλαγή της GELU που χρησιμοποιεί το GPT-2)
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model), κάνε προβολή πίσω στην αρχική διάσταση
        return x


# Δοκίμασέ το

print("Ορίστηκε η κλάση FeedForward!")

In [ ]:
# ===== ΜΠΛΟΚ TRANSFORMER (από το Κεφάλαιο 10) =====
class TransformerBlock(nn.Module):
    """
    Πλήρες μπλοκ Transformer με προσοχή πολλαπλών κεφαλών, δίκτυο πρόσθιας τροφοδότησης,
    υπολειμματικές συνδέσεις και κανονικοποίηση επιπέδου (πριν από κάθε υποεπίπεδο, όπως στο GPT-2).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Ορίσματα:
            d_model: Διάσταση του μοντέλου (768)
            num_heads: Αριθμός κεφαλών προσοχής (12)
            d_ff: Ενδιάμεση διάσταση του δικτύου πρόσθιας τροφοδότησης (3072 = 4 * d_model)
            dropout: Πιθανότητα απενεργοποίησης (dropout)
        """
        super().__init__()

        # Κανονικοποίηση επιπέδου (πριν από κάθε υποεπίπεδο)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Προσοχή πολλαπλών κεφαλών
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)

        # Δίκτυο πρόσθιας τροφοδότησης
        self.ffn = FeedForward(d_model, d_ff, dropout)

        # Dropout (εφαρμόζεται μετά από κάθε υποεπίπεδο)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Ορίσματα:
            x: Διανυσματικές αναπαραστάσεις εισόδου (batch, seq, d_model)
            mask: Αιτιακή μάσκα (seq, seq) ή (1, 1, seq, seq)

        Επιστρέφει:
            x: Διανυσματικές αναπαραστάσεις εξόδου (batch, seq, d_model)
            attn_weights: Βάρη προσοχής (batch, heads, seq, seq)
        """
        # ===== Προσοχή πολλαπλών κεφαλών με υπολειμματική σύνδεση =====
        # Προκανονικοποίηση: Κανονικοποίησε την είσοδο ΠΡΙΝ από τον μηχανισμό προσοχής
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Υπολειμματική σύνδεση

        # ===== Δίκτυο πρόσθιας τροφοδότησης με υπολειμματική σύνδεση =====
        # Προκανονικοποίηση: Κανονικοποίησε την είσοδο ΠΡΙΝ από το FFN
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Υπολειμματική σύνδεση

        return x, attn_weights


# Δοκίμασε ένα πλήρες μπλοκ Transformer

print("Το TransformerBlock ορίστηκε!")

In [ ]:
# ===== ΡΥΘΜΙΣΕΙΣ GPT (από το κεφάλαιο 11) =====
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """Ρυθμίσεις για το μοντέλο MiniGPT."""
    vocab_size: int = 50257      # Μέγεθος λεξιλογίου του GPT-2
    max_seq_len: int = 1024      # Μέγιστο μήκος συμφραζομένων
    embed_dim: int = 768         # Διάσταση της διανυσματικής αναπαράστασης (d_model)
    num_heads: int = 12          # Αριθμός κεφαλών προσοχής
    num_layers: int = 12         # Αριθμός μπλοκ Transformer
    d_ff: int = 3072             # Διάσταση του κρυφού επιπέδου του δικτύου πρόσθιας τροφοδότησης (4 * embed_dim)
    dropout: float = 0.1         # Πιθανότητα dropout

    def __post_init__(self):
        """Έλεγξε αν οι ρυθμίσεις είναι έγκυρες μετά την αρχικοποίηση."""
        assert self.embed_dim % self.num_heads == 0, \
            f"Η τιμή embed_dim ({self.embed_dim}) πρέπει να διαιρείται ακριβώς με την τιμή " \
            f"num_heads ({self.num_heads})"


In [ ]:
# ===== ΜΟΝΤΕΛΟ MINIGPT (από το κεφάλαιο 11) =====
class MiniGPT(nn.Module):
    """
    Ένα γλωσσικό μοντέλο τύπου GPT με μόνο τα βασικά στοιχεία.
    Συνδυάζει διανυσματικές αναπαραστάσεις, μπλοκ Transformer και μια κεφαλή γλωσσικής μοντελοποίησης.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # ===== Διανυσματικές αναπαραστάσεις για τα token και τις θέσεις τους (κεφάλαιο 9) =====
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # ===== Μπλοκ Transformer (κεφάλαιο 10) =====
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # ===== Τελικό επίπεδο LayerNorm =====
        self.ln_f = nn.LayerNorm(config.embed_dim)

        # ===== Κεφαλή γλωσσικής μοντελοποίησης =====
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # ===== Κοινή χρήση βαρών =====
        # Χρησιμοποίησε τα ίδια βάρη για τις διανυσματικές αναπαραστάσεις και την προβολή εξόδου
        self.lm_head.weight = self.token_embed.weight

        # Αρχικοποίησε τα βάρη
        self._init_weights()

    def _init_weights(self):
        """Αρχικοποίησε τα βάρη με μικρές τυχαίες τιμές."""
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        """
        Πρόσθιο πέρασμα του μοντέλου.

        Ορίσματα:
            token_ids: Αναγνωριστικά των token εισόδου (batch, seq)
            return_attention: Αν θα επιστραφούν τα βάρη προσοχής

        Επιστρέφει:
            logits: Βαθμολογίες για τα token του λεξιλογίου (batch, seq, vocab_size)
            attention_weights: Προαιρετική λίστα με βάρη προσοχής ανά μπλοκ
        """
        batch, seq = token_ids.shape
        device = token_ids.device

        # ===== Διανυσματικές αναπαραστάσεις =====
        # Διανυσματικές αναπαραστάσεις των token
        tok_emb = self.token_embed(token_ids)  # (batch, seq, embed_dim)

        # Διανυσματικές αναπαραστάσεις των θέσεων
        positions = torch.arange(seq, device=device)  # (seq,)
        pos_emb = self.pos_embed(positions)  # (seq, embed_dim)

        # Συνδύασε τις αναπαραστάσεις και εφάρμοσε dropout
        x = self.dropout(tok_emb + pos_emb)  # (batch, seq, embed_dim)

        # ===== Δημιούργησε αιτιακή μάσκα =====
        # Κάτω τριγωνική μάσκα: κάθε θέση δίνει προσοχή στην ίδια και στις προηγούμενες θέσεις, όχι στις επόμενες.
        mask = torch.tril(torch.ones(seq, seq, device=device))

        # ===== Πέρασμα από τα μπλοκ του Transformer =====
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        # ===== Τελικό LayerNorm =====
        x = self.ln_f(x)  # (batch, seq, embed_dim)

        # ===== Κεφαλή γλωσσικής μοντελοποίησης =====
        logits = self.lm_head(x)  # (batch, seq, vocab_size)

        if return_attention:
            return logits, attention_weights
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        Δημιούργησε token που συνεχίζουν την ακολουθία input_ids (η έξοδος περιλαμβάνει την προτροπή).

        Ορίσματα:
            input_ids: Αναγνωριστικά των token της προτροπής (batch, seq)
            max_new_tokens: Πόσα νέα token θα δημιουργηθούν
            temperature: Θερμοκρασία δειγματοληψίας (χρησιμοποιείται μόνο όταν do_sample=True)
            do_sample: Δειγματοληψία από την κατανομή· False = επιλογή του πιθανότερου token σε κάθε βήμα (argmax)
            eos_token_id: Σταμάτησε νωρίτερα αν κάθε ακολουθία δημιουργήσει αυτό το token

        Επιστρέφει:
            Αναγνωριστικά των token (batch, seq + new tokens)
        """
        self.eval()

        # Οι διανυσματικές αναπαραστάσεις των θέσεων καλύπτουν μόνο max_seq_len θέσεις
        limit = self.config.max_seq_len
        if input_ids.shape[1] + max_new_tokens > limit:
            raise ValueError(
                f"Προτροπή ({input_ids.shape[1]} token) + max_new_tokens "
                f"({max_new_tokens}): υπέρβαση του ορίου των {limit} token για το "
                "πλαίσιο του μοντέλου. Συντόμευσε την προτροπή ή μείωσε την τιμή του max_new_tokens.")
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids


## 2. Φόρτωσε το βασικό μοντέλο του Κεφαλαίου 12 (εισαγωγή αποθηκευμένης κατάστασης εκπαίδευσης — checkpoint)

Το βιβλίο κάνει προσαρμογή με περαιτέρω εκπαίδευση ξεκινώντας από την **αποθηκευμένη κατάσταση εκπαίδευσης του μοντέλου Shakespeare του Κεφαλαίου 12**. Το ίδιο κάνει και αυτό το σημειωματάριο. Η αρχική αξιολόγηση, η εκπαίδευση με LoRA και η επαναφόρτωση του προσαρμογέα LoRA χρησιμοποιούν όλες **τα ίδια βασικά βάρη**. Μπορείς να τα αποκτήσεις με δύο τρόπους:

- **Τρόπος Α (προτιμότερος):** Αντέγραψε μόνο το δικό σου `best_model.pt` από το Κεφάλαιο 12 στον κατάλογο εργασίας του πυρήνα (`%pwd`) πριν εκτελέσεις το επόμενο κελί. Στο Colab, ανέβασε το αρχείο από το πλαίσιο Files ή αφαίρεσε το σύμβολο σχολίου από τις δύο γραμμές για τη μεταφόρτωση στο παρακάτω κελί. Μην φορτώνεις αρχεία από άγνωστες πηγές.
- **Τρόπος Β (νέα συνεδρία Colab, χωρίς αρχείο):** Το παρακάτω κελί εκτελεί απευθείας τα βήματα εκπαίδευσης του Κεφαλαίου 12, κάνοντας μια **σύντομη εκπαίδευση μίας εποχής** στο TinyShakespeare, και αποθηκεύει το αποτέλεσμα ως `best_model.pt`. Αυτό το βασικό μοντέλο είναι πιο αδύναμο από εκείνο που προκύπτει μετά την πλήρη εκπαίδευση τριών εποχών του βιβλίου. Έτσι, οι τιμές απώλειας που θα δεις θα διαφέρουν από αυτές του βιβλίου, αλλά όλα τα παρακάτω κελιά χρησιμοποιούν το ίδιο βασικό μοντέλο.

Το βασικό μοντέλο με τυχαία αρχικά βάρη χρησιμοποιείται μόνο σε ένα *ξεχωριστό πείραμα που επισημαίνεται ρητά* στο τέλος του σημειωματαρίου. Δεν είναι ποτέ η προεπιλογή.

In [ ]:
# ===== ΕΙΣΑΓΩΓΗ ΑΠΟΘΗΚΕΥΜΕΝΗΣ ΚΑΤΑΣΤΑΣΗΣ ΕΚΠΑΙΔΕΥΣΗΣ ΤΟΥ ΚΕΦΑΛΑΙΟΥ 12 =====
# Κάθε κελί παρακάτω (αρχική αξιολόγηση, εκπαίδευση LoRA, επαναφόρτωση) χρησιμοποιεί ΑΥΤΟ το βασικό μοντέλο.
import os
import urllib.request

config = GPTConfig(
    vocab_size=50257, max_seq_len=128, embed_dim=256,
    num_heads=4, num_layers=4, d_ff=1024, dropout=0.1
)

# Φόρτωσε τον tokenizer. Με use_fast=True παίρνουμε τον GPT2TokenizerFast, που
# μας δίνει τις θέσεις των token στο κείμενο, μετρημένες σε χαρακτήρες, για τη μάσκα απώλειας παρακάτω.
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# Επιλογή Α: ανέβασε τη ΔΙΚΗ ΣΟΥ αποθηκευμένη κατάσταση εκπαίδευσης από το Κεφάλαιο 12 (επιλογή αρχείου στο Colab):
# from google.colab import files
# files.upload()  # επίλεξε το best_model.pt

if not os.path.exists("best_model.pt"):
    # Επιλογή Β: αν δεν έχεις αποθηκευμένη κατάσταση εκπαίδευσης -> σύντομη εκπαίδευση του Κεφαλαίου 12 (1 εποχή).
    # Ίδια διαδικασία με το σημειωματάριο του Κεφαλαίου 12: τμήματα των 512 χαρακτήρων, παρτίδες των 32,
    # AdamW με γραμμική προθέρμανση και μάσκα προσοχής για να αγνοούνται τα token συμπλήρωσης.
    print("Δεν βρέθηκε το best_model.pt: ξεκινά η σύντομη εκπαίδευση του Κεφαλαίου 12 (1 εποχή)...")
    from transformers import get_linear_schedule_with_warmup

    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        "shakespeare.txt")
    text = open("shakespeare.txt").read()

    def create_lm_labels(token_ids):
        inputs = token_ids[:, :-1]
        targets = token_ids[:, 1:].clone()
        return inputs, targets  # τα token συμπλήρωσης αγνοούνται μέσα στον βρόχο με τη βοήθεια της attention_mask

    class TextDataset(Dataset):
        def __init__(self, text, chunk_size=512):
            self.chunks = [text[i:i + chunk_size]
                           for i in range(0, len(text) - chunk_size + 1, chunk_size)]
        def __len__(self):
            return len(self.chunks)
        def __getitem__(self, idx):
            return self.chunks[idx]

    def ch12_collate(batch, tokenizer, max_length=128):
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt")
        return enc["input_ids"], enc["attention_mask"]

    ds = TextDataset(text, chunk_size=512)
    n_train = int(0.9 * len(ds))
    train_ds, val_ds = torch.utils.data.random_split(
        ds, [n_train, len(ds) - n_train],
        generator=torch.Generator().manual_seed(42))
    collate12 = partial(ch12_collate, tokenizer=tokenizer, max_length=128)
    train_loader12 = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate12)

    set_seed(42)
    model = MiniGPT(config).to(device)
    optimizer = AdamW(model.parameters(), lr=3e-4)
    # Εδώ μία εποχή έχει μόνο ~62 ενημερώσεις: κράτησε τα βήματα προθέρμανσης λιγότερα από το σύνολο, ώστε
    # ο ρυθμός μάθησης να αρχίσει όντως να μειώνεται αντί να αυξάνεται σε όλη τη διάρκεια της εκπαίδευσης.
    total_steps12 = len(train_loader12)
    warmup_steps12 = min(100, max(1, total_steps12 // 10))
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps12,
        num_training_steps=total_steps12)

    model.train()
    total_loss = 0
    for input_ids, attention_mask in tqdm(train_loader12, desc="Σύντομη εκπαίδευση του Κεφαλαίου 12"):
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # αγνόησε τα token συμπλήρωσης στον υπολογισμό της απώλειας
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                               targets.reshape(-1), ignore_index=-100)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Η σύντομη εκπαίδευση ολοκληρώθηκε: μέση απώλεια {total_loss / len(train_loader12):.4f}")
    # Αρχείο μόνο με βάρη: δεν περιέχει την κατάσταση του βελτιστοποιητή ή της RNG και μπορείς να το ξαναφορτώσεις με ασφάλεια οπουδήποτε
    torch.save({"model_state_dict": model.state_dict()}, "best_model.pt")
    print("Αποθηκεύτηκε το best_model.pt (βασικό μοντέλο μετά από σύντομη εκπαίδευση 1 εποχής· πιο αδύναμο από το μοντέλο του βιβλίου που εκπαιδεύτηκε για 3 εποχές)")

# Όποια επιλογή κι αν ακολούθησες, φόρτωσε την αποθηκευμένη κατάσταση εκπαίδευσης στο ΙΔΙΟ βασικό μοντέλο που χρησιμοποιείται παντού παρακάτω.
# Η αποθηκευμένη κατάσταση για τη συνέχιση της εκπαίδευσης του Κεφαλαίου 12 περιέχει και καταστάσεις RNG (αντικείμενα NumPy που
# απορρίπτονται κατά την ασφαλή φόρτωση), οπότε επίτρεψε μόνο τις συγκεκριμένες κλάσεις και καμία άλλη.
import numpy as np
torch.serialization.add_safe_globals([np._core.multiarray._reconstruct,
                                      np.ndarray, np.dtype])
base_model = MiniGPT(config).to(device)
try:
    checkpoint = torch.load("best_model.pt", map_location=device, weights_only=True)
except Exception:
    # Οι παλιότερες αποθηκευμένες καταστάσεις εκπαίδευσης μπορεί να περιέχουν κι άλλους τύπους καταστάσεων NumPy. Αυτό το αρχείο
    # προέρχεται από εκπαίδευση που έκανες ΕΣΥ στο Κεφάλαιο 12, οπότε είναι αποδεκτό να το φορτώσεις ολόκληρο.
    checkpoint = torch.load("best_model.pt", map_location=device, weights_only=False)
base_model.load_state_dict(checkpoint["model_state_dict"])
base_model.eval()
BASE_SOURCE = "Αποθηκευμένη κατάσταση εκπαίδευσης του κεφαλαίου 12 (best_model.pt)"
print(f"Το βασικό μοντέλο είναι έτοιμο: {sum(p.numel() for p in base_model.parameters()):,} παράμετροι")
print(f"Βάρη του βασικού μοντέλου: {BASE_SOURCE}")

## 3. Ετοίμασε το σύνολο δεδομένων για προσαρμογή με περαιτέρω εκπαίδευση

Φτιάξε το σύνολο συχνών ερωτήσεων και απαντήσεων της TechStartup στα αγγλικά, όπως στο κεφάλαιο. Με αρχική τιμή 42 για τη γεννήτρια τυχαίων αριθμών, οι πέντε βασικές διατυπώσεις που κρατάμε για αξιολόγηση καλύπτουν τέσσερις από τις επτά προθέσεις που χρησιμοποιούμε στην εκπαίδευση. Έτσι ελέγχουμε αν το μοντέλο δίνει γνωστές απαντήσεις σε νέες διατυπώσεις, όχι αν αναγνωρίζει νέες προθέσεις. Τα παραδείγματα και οι προτροπές μένουν στα αγγλικά, ώστε ο tokenizer να δέχεται τις ίδιες εισόδους και η εξαγωγή FAQ να συμφωνεί με το αρχείο αναφοράς. Για τον ίδιο λόγο διατηρούμε στα αγγλικά τις τέσσερις γραμμές εξόδου που περιγράφουν τον διαχωρισμό.

In [ ]:
# ===== ΣΥΝΟΛΟ ΔΕΔΟΜΕΝΩΝ FAQ (όπως στο βιβλίο: χωρισμός ανά ΟΜΑΔΑ παραφράσεων) =====
import json
import random

# Επτά ζεύγη συχνών ερωτήσεων και απαντήσεων που γράψαμε στο χέρι για τη φανταστική μας εταιρεία
base_examples = [
    {"instruction": "What does TechStartup Inc do?",
     "response": "TechStartup Inc builds AI-powered tools."},

    {"instruction": "When was TechStartup Inc founded?",
     "response": "TechStartup Inc was founded in 2020."},

    {"instruction": "Where is TechStartup Inc located?",
     "response": "TechStartup Inc is in Amsterdam."},

    {"instruction": "What is SmartScheduler?",
     "response": "SmartScheduler is our AI calendar assistant."},

    {"instruction": "How much does SmartScheduler cost?",
     "response": "10 euros/month individual, 8 euros/user teams."},

    {"instruction": "How do I reset my password?",
     "response": "Click 'Forgot Password' on the login page."},

    {"instruction": "Can I export my data?",
     "response": "Yes, go to Settings > Data > Export."},
]


# Εναλλακτικές διατυπώσεις για κάθε ερώτηση (με την ίδια σειρά όπως στο base_examples)
paraphrases = [
    ["What does TechStartup Inc do?", "What is TechStartup Inc's business?",
     "Tell me what TechStartup Inc makes.", "What kind of company is TechStartup Inc?"],
    ["When was TechStartup Inc founded?", "What year was TechStartup Inc started?",
     "How old is TechStartup Inc?", "When did TechStartup Inc begin?"],
    ["Where is TechStartup Inc located?", "Where is TechStartup Inc based?",
     "Which city is TechStartup Inc in?", "Where are TechStartup Inc's offices?"],
    ["What is SmartScheduler?", "Tell me about SmartScheduler.",
     "What does SmartScheduler do?", "Describe SmartScheduler."],
    ["How much does SmartScheduler cost?", "What is SmartScheduler's price?",
     "What does SmartScheduler cost per month?", "How much is a SmartScheduler subscription?"],
    ["How do I reset my password?", "I forgot my password, what do I do?",
     "How can I change my password?", "What is the password reset process?"],
    ["Can I export my data?", "How do I export my data?",
     "Is data export possible?", "Where is the data export option?"],
]

def expand_faq(base_examples, paraphrases, seed=42):
    """Δημιούργησε περισσότερα ζεύγη ερωτήσεων και απαντήσεων με τρόπο που δίνει πάντα το ίδιο αποτέλεσμα, χωρίζοντάς τα ανά ΟΜΑΔΑ παραφράσεων.

    Κάθε βασική διατύπωση δίνει μια ομάδα τεσσάρων παραδειγμάτων: την ίδια τη διατύπωση και τρεις παραλλαγές με ευγενική εισαγωγή
    (7 ερωτήσεις x 4 παραφράσεις = 28 ομάδες των 4). Ανακατεύουμε τις ΟΜΑΔΕΣ
    και κρατάμε 5 ολόκληρες ομάδες (20 παραδείγματα) για αξιολόγηση ΠΡΙΝ δημιουργήσουμε τα κείμενα
    των δύο συνόλων. Έτσι, καμία διατύπωση από το σύνολο αξιολόγησης, με ή χωρίς ευγενική εισαγωγή, δεν
    χρησιμοποιήθηκε στην εκπαίδευση. Για την εκπαίδευση παίρνουμε 80 από τα υπόλοιπα 92 παραδείγματα.
    """
    prefixes = ["", "Please tell me: ", "Quick question: ", "Could you tell me: "]
    groups = []
    for qid, (example, variants) in enumerate(zip(base_examples, paraphrases)):
        for paraphrase in variants:
            groups.append((qid, paraphrase,
                           [{"instruction": p + paraphrase,
                             "response": example["response"]} for p in prefixes]))
    rng = random.Random(seed)
    rng.shuffle(groups)

    # Κράτησε 5 ολόκληρες ομάδες για αξιολόγηση, αλλά ποτέ την ΤΕΛΕΥΤΑΙΑ ομάδα μιας ερώτησης:
    # Για κάθε πρόθεση, κράτησε τουλάχιστον μία διατύπωση στα δεδομένα εκπαίδευσης.
    left = {qid: sum(1 for q, _, _ in groups if q == qid)
            for qid in range(len(base_examples))}
    eval_groups, train_groups = [], []
    for qid, paraphrase, examples in groups:
        if len(eval_groups) < 5 and left[qid] > 1:
            eval_groups.append((qid, paraphrase, examples))
        else:
            train_groups.append((qid, paraphrase, examples))
        left[qid] -= 1

    eval_data = [ex for _, _, g in eval_groups for ex in g]      # 5 x 4 = 20
    train_pool = [ex for _, _, g in train_groups for ex in g]    # 23 x 4 = 92
    rng.shuffle(train_pool)
    train_data = train_pool[:80]
    return train_data, eval_data

train_data, eval_data = expand_faq(base_examples, paraphrases)

# Έλεγχος 1: καμία ερώτηση δεν έχει ακριβώς το ίδιο κείμενο και στα δύο σύνολα (ο κλασικός έλεγχος για διαρροή δεδομένων)
train_questions = {ex["instruction"] for ex in train_data}
assert all(ex["instruction"] not in train_questions for ex in eval_data)

# Έλεγχος 2: τα δύο σύνολα δεν έχουν κοινές ΟΜΑΔΕΣ παραφράσεων. Αφαίρεσε τις εισαγωγικές φράσεις ευγένειας και
# έλεγξε ότι καμία διατύπωση της αξιολόγησης δεν εμφανίζεται στην εκπαίδευση με ΟΠΟΙΑΔΗΠΟΤΕ εισαγωγική φράση.
def core_question(instruction):
    for p in ["Please tell me: ", "Quick question: ", "Could you tell me: "]:
        if instruction.startswith(p):
            return instruction[len(p):]
    return instruction

train_cores = {core_question(q) for q in train_questions}
assert all(core_question(ex["instruction"]) not in train_cores for ex in eval_data)

# Εδώ κάθε διαφορετική απάντηση αντιστοιχεί σε μία πρόθεση στις συχνές ερωτήσεις. Μέτρησε πόσες προθέσεις καλύπτει το καθένα από τα δύο σύνολα.
train_intents = {ex["response"] for ex in train_data}
eval_intents = {ex["response"] for ex in eval_data}
print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")
print(f"Intents covered in training: {len(train_intents)}/7")
print(f"Intents covered in eval: {len(eval_intents)}/7; "
      f"also in training: {len(eval_intents & train_intents)}")
print(f"Eval phrasings seen in training (prefixes stripped): "
      f"{sum(core_question(ex['instruction']) in train_cores for ex in eval_data)}/20")
# Output: Train: 80, Eval: 20
# Output: Intents covered in training: 7/7
# Output: Intents covered in eval: 4/7; also in training: 4
# Output: Eval phrasings seen in training (prefixes stripped): 0/20


def save_as_jsonl(examples, filepath):
    """Αποθήκευσε τα παραδείγματα σε μορφή JSONL (ένα αντικείμενο JSON ανά γραμμή)."""
    with open(filepath, 'w') as f:
        for example in examples:
            f.write(json.dumps(example) + '\n')

def load_jsonl(filepath):
    """Φόρτωσε τα παραδείγματα από αρχείο σε μορφή JSONL."""
    examples = []
    with open(filepath, 'r') as f:
        for line in f:
            examples.append(json.loads(line))
    return examples

# Γράψε πρώτα τις 80 γραμμές εκπαίδευσης και μετά τις 20 γραμμές αξιολόγησης. Το αγγλικό αρχείο
# αναφοράς που περιλαμβάνεται στο πακέτο για αυτόν τον σπόρο τυχαιότητας είναι το data/faq_dataset.jsonl.
save_as_jsonl(train_data + eval_data, 'faq_dataset.jsonl')


In [ ]:
# ===== ΜΟΡΦΟΠΟΙΗΣΗ + ΟΜΑΔΟΠΟΙΗΣΗ ΣΕ ΠΑΡΤΙΔΕΣ (από το βιβλίο) =====
# Όρισε τα ειδικά token
INST_START = "[INST]"
INST_END = "[/INST]"

def build_prompt(instruction):
    """Η ΜΙΑ κοινή μορφή προτροπής για την εκπαίδευση και την παραγωγή κειμένου.

    Τελειώνει αμέσως μετά το [/INST], χωρίς τελικό κενό: ένα τελικό κενό
    μπορεί να δώσει διαφορετικά token όταν είναι μόνο του απ’ ό,τι όταν ενώνεται
    με την πρώτη λέξη της απάντησης (" Tech"). Έτσι, κατά την παραγωγή κειμένου,
    το μοντέλο θα έπαιρνε μια αρχική ακολουθία που δεν είχε δει ποτέ στην εκπαίδευση.
    """
    return f"{INST_START} {instruction} {INST_END}"

def format_example(instruction, response, tokenizer):
    """
    Μορφοποίησε ένα ζεύγος οδηγίας/απάντησης για την εκπαίδευση.

    Χρειάζεται γρήγορο tokenizer (GPT2TokenizerFast, που φορτώνεται στη δική μας ρύθμιση):
    μόνο οι γρήγοροι tokenizer δίνουν τις θέσεις των χαρακτήρων όπου αρχίζει και τελειώνει κάθε token.

    Επιστρέφει:
        input_ids: Ολόκληρη η ακολουθία σε token
        labels: Όπως το input_ids, αλλά με -100 για τα token της οδηγίας
    """
    # Μορφή: [INST] οδηγία [/INST] απάντηση + τέλος κειμένου.
    # Το token EOS μαθαίνει στο μοντέλο πού ΤΕΛΕΙΩΝΟΥΝ οι απαντήσεις, ώστε να μπορεί
    # να σταματά εκεί αντί να συνεχίζει να γράφει άσκοπα μετά την απάντηση.
    prompt = build_prompt(instruction)
    full_text = prompt + " " + response + tokenizer.eos_token

    # Χώρισε το full_text σε token και ζήτησε για κάθε token το εύρος των χαρακτήρων που καλύπτει
    enc = tokenizer(full_text, return_offsets_mapping=True)
    input_ids = torch.tensor(enc["input_ids"])

    # Βρες πού αρχίζει η απάντηση: στο πρώτο token του οποίου όλοι οι
    # χαρακτήρες βρίσκονται μετά την προτροπή. Το BPE ενώνει το κενό που βάλαμε
    # με την πρώτη λέξη της απάντησης (" Tech"), οπότε αυτό το token ανήκει
    # στην απάντηση· αν μετρούσες μόνο τα token της προτροπής, θα το κάλυπτες με τη μάσκα.
    response_start = next(i for i, (_, end) in enumerate(enc["offset_mapping"])
                          if end > len(prompt))

    # Φτιάξε τις ετικέτες: βάλε -100 στα token της οδηγίας (δεν μετρούν στην απώλεια)
    labels = input_ids.clone()
    labels[:response_start] = -100  # Κάλυψε τα token της οδηγίας με τη μάσκα απώλειας

    return input_ids, labels


import torch
from functools import partial
from torch.utils.data import DataLoader

PAD_ID = tokenizer.eos_token_id  # Το GPT-2 συμπληρώνει τις ακολουθίες με το token τέλους κειμένου

def collate_fn(examples, tokenizer, max_length=128):
    """Χώρισε τα ζεύγη οδηγίας/απάντησης σε token και φτιάξε input_ids και labels με συμπλήρωση στο ίδιο μήκος."""
    batch_ids, batch_labels = [], []
    for ex in examples:
        ids, labels = format_example(ex['instruction'], ex['response'], tokenizer)
        ids, labels = ids[:max_length], labels[:max_length]
        pad = max_length - len(ids)
        ids = torch.cat([ids, torch.full((pad,), PAD_ID, dtype=torch.long)])
        labels = torch.cat([labels, torch.full((pad,), -100, dtype=torch.long)])
        batch_ids.append(ids)
        batch_labels.append(labels)
    return {'input_ids': torch.stack(batch_ids),
            'labels': torch.stack(batch_labels)}

collate = partial(collate_fn, tokenizer=tokenizer)
train_loader = DataLoader(train_data, batch_size=8, shuffle=True, collate_fn=collate)


# Έλεγξε μία παρτίδα
batch = next(iter(train_loader))
print(f"Διαστάσεις του τανυστή αναγνωριστικών εισόδου: {batch['input_ids'].shape}")
print(f"Διαστάσεις ετικετών: {batch['labels'].shape}")
print(f"Θέσεις με μάσκα στο πρώτο παράδειγμα: {(batch['labels'][0] == -100).sum().item()}")

## 4. Αρχική αξιολόγηση (ΠΡΙΝ από την προσαρμογή με περαιτέρω εκπαίδευση)

Ας δούμε πώς απαντά το βασικό μοντέλο στις προτροπές μας για τις συχνές ερωτήσεις.

In [ ]:
@torch.no_grad()
def generate_response(model, prompt, tokenizer, max_new_tokens=50, temperature=0.8):
    """Δημιούργησε μια απάντηση σε μια προτροπή που περιέχει οδηγία."""
    model.eval()
    full_prompt = build_prompt(prompt)
    input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(device)
    
    output_ids = model.generate(
        input_ids, 
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True
    )
    
    response = tokenizer.decode(output_ids[0][len(input_ids[0]):])
    return response.strip()

# Δοκίμασε το μοντέλο με προτροπές για συχνές ερωτήσεις
test_prompts = [
    "What does TechStartup Inc do?",
    "How do I reset my password?",
    "What is SmartScheduler?"
]

print(f"ΠΡΙΝ ΑΠΟ ΤΗΝ ΠΕΡΑΙΤΕΡΩ ΕΚΠΑΙΔΕΥΣΗ ({BASE_SOURCE}):")
print("="*60)
for prompt in test_prompts:
    response = generate_response(base_model, prompt, tokenizer)
    print(f"\nΕρ.: {prompt}")
    print(f"Απ.: {response[:100]}...")
print("\n(Ασυναρτησίες σε ύφος Σαίξπηρ - το βασικό μοντέλο του Κεφαλαίου 12 δεν ξέρει")
print("τίποτα για την TechStartup Inc, κι ένα μοντέλο 16 εκατομμυρίων παραμέτρων που εκπαιδεύτηκε από το μηδέν")
print("δεν έμαθε ποτέ να γράφει με ευχέρεια στα αγγλικά. Ας έχουμε ρεαλιστικές προσδοκίες!)")

## 5. Όρισε τα μέρη του LoRA

Φτιάξε το επίπεδο του προσαρμογέα LoRA, που προσθέτει εκπαιδεύσιμες παραμέτρους χωρίς να αλλάζει τα βάρη του βασικού μοντέλου.

### Γιατί LoRA αντί για πλήρη προσαρμογή με περαιτέρω εκπαίδευση;

Η πλήρης προσαρμογή με περαιτέρω εκπαίδευση έχει ένα πρόβλημα: την **καταστροφική λήθη**. Όταν ενημερώνεις όλα τα βάρη για μια νέα εργασία, το μοντέλο μπορεί να «ξεχάσει» όσα έμαθε κατά την προεκπαίδευση. Το LoRA περιορίζει πολύ αυτό το πρόβλημα: κρατά τα βάρη του βασικού μοντέλου *παγωμένα* και προσθέτει μικρούς εκπαιδεύσιμους πίνακες *δίπλα* τους.

Σκέψου τα γυαλιά διαβάσματος: τα μάτια σου (το βασικό μοντέλο) παραμένουν ίδια, αλλά τα γυαλιά (το LoRA) προσφέρουν μια μικρή διόρθωση για συγκεκριμένες εργασίες.

In [ ]:
# ===== ΓΡΑΜΜΙΚΟ ΕΠΙΠΕΔΟ LORA (σύμβαση της δημοσίευσης: A για μείωση διαστάσεων, B για αύξηση) =====
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    """
    Ένα γραμμικό επίπεδο με προσαρμογή χαμηλού βαθμού.

    Φαντάσου ότι προσθέτεις ένα μικρό "σημείωμα με διορθώσεις" σε ένα υπάρχον επίπεδο.
    Το βασικό επίπεδο διατηρεί τα βάρη του παγωμένα, ενώ δύο μικροί πίνακες μαθαίνουν τις διορθώσεις.
    """

    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()

        # Το αρχικό επίπεδο (θα παγώσουμε τα βάρη του)· η ρύθμιση του bias είναι ίδια με εκείνη του qkv_proj του GPT-2
        self.base = nn.Linear(in_features, out_features)

        # Πίνακες LoRA (σύμβαση της δημοσίευσης: A για μείωση διαστάσεων, B για αύξηση)
        # A: (in_features, rank) - προβολή σε χώρο "λιγότερων" διαστάσεων
        # B: (rank, out_features) - προβολή σε χώρο "περισσότερων" διαστάσεων
        self.lora_A = nn.Parameter(torch.zeros(in_features, rank))
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))

        # Αρχικοποίησε τον A με μικρές τυχαίες τιμές (Kaiming) και τον B με μηδενικά.
        # Κάνε την αρχικοποίηση μέσω της μεταθεμένης όψης ώστε η μέθοδος Kaiming να βλέπει (rank, in_features),
        # δηλαδή το σχήμα των βαρών του nn.Linear(in_features, rank): έτσι το fan_in γίνεται
        # in_features. Αν γράψεις απευθείας στην όψη, ενημερώνεται το ίδιο το lora_A.
        nn.init.kaiming_uniform_(self.lora_A.T, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

        # Συντελεστής κλιμάκωσης
        self.scale = alpha / rank

    def freeze_base(self):
        """Πάγωσε το βασικό επίπεδο ώστε να εκπαιδεύονται μόνο οι πίνακες LoRA."""
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

    def forward(self, x):
        # Σχήμα του x: (batch, sequence, in_features)

        # Αρχική έξοδος από τα παγωμένα βάρη
        base_output = self.base(x)  # (batch, seq, out_features)

        # Διαδρομή LoRA: x @ A @ B, με κλιμάκωση
        # x @ A: (batch, seq, in_features) @ (in_features, rank)
        #      = (batch, seq, rank)  [συμπίεση!]
        # ... @ B: (batch, seq, rank) @ (rank, out_features)
        #        = (batch, seq, out_features)  [επέκταση στην πλήρη διάσταση εξόδου]
        lora_output = (x @ self.lora_A @ self.lora_B) * self.scale

        return base_output + lora_output

print("Ορίστηκε το LoRALinear!")

# Δείξε πόσες παραμέτρους γλιτώνεις από την εκπαίδευση
in_features, out_features, rank = 256, 256, 8
full_params = in_features * out_features
lora_params = in_features * rank + rank * out_features

print(f"\nΣύγκριση παραμέτρων (επίπεδο 256x256):")
print(f"  Πλήρης περαιτέρω εκπαίδευση: {full_params:,} παράμετροι")
print(f"  LoRA (βαθμός=8):    {lora_params:,} παράμετροι")
print(f"  Εξοικονόμηση:          {full_params/lora_params:.1f} φορές λιγότερες παράμετροι!")

## 6. Εφάρμοσε το LoRA στο μοντέλο

Αντικατάστησε τις προβολές του μηχανισμού προσοχής με εκδοχές LoRA και πάγωσε το βασικό μοντέλο.

In [ ]:
def add_lora_to_model(model, rank=8, alpha=16):
    """
    Πρόσθεσε LoRA στις προβολές του μηχανισμού προσοχής.

    Εδώ εφαρμόζουμε το LoRA στο qkv_proj, την ενιαία προβολή για ερωτήματα, κλειδιά και τιμές.
    Ο προσαρμογέας LoRA μαθαίνει από κοινού τις διορθώσεις για τα Q, K και V.
    """
    # Βήμα 1: Πάγωσε πρώτα ΟΛΕΣ τις παραμέτρους
    for param in model.parameters():
        param.requires_grad = False

    # Βήμα 2: Αντικατάστησε το qkv_proj σε κάθε μπλοκ του transformer
    # Περνάμε ένα προς ένα από τα μπλοκ που φτιάξαμε στο κεφάλαιο 11
    for block in model.blocks:
        # Πάρε την προβολή QKV του επιπέδου προσοχής
        original_qkv = block.attn.qkv_proj
        in_features = original_qkv.in_features
        out_features = original_qkv.out_features

        # Δημιούργησε το νέο επίπεδο με LoRA στην ίδια συσκευή με το μοντέλο
        lora_qkv = LoRALinear(in_features, out_features, rank, alpha)
        lora_qkv = lora_qkv.to(original_qkv.weight.device)

        # Αντίγραψε τα προεκπαιδευμένα βάρη και τον όρο μετατόπισης στο βασικό επίπεδο.
        lora_qkv.base.weight.data = original_qkv.weight.data.clone()
        lora_qkv.base.bias.data = original_qkv.bias.data.clone()

        # Πάγωσε το βασικό επίπεδο (οι πίνακες LoRA παραμένουν εκπαιδεύσιμοι)
        lora_qkv.freeze_base()
        if lora_qkv.base.bias is not None:
            lora_qkv.base.bias.requires_grad = False

        # Αντικατάστησε το επίπεδο
        block.attn.qkv_proj = lora_qkv

    return model


# Ίδια βάση με το βιβλίο: η αποθηκευμένη κατάσταση εκπαίδευσης του κεφαλαίου 12 που εισαγάγαμε παραπάνω.
# Η αρχική αξιολόγηση, η εκπαίδευση και η επαναφόρτωση χρησιμοποιούν τα ίδια ακριβώς βάρη, οπότε οι
# συγκρίσεις ΠΡΙΝ/ΜΕΤΑ είναι δίκαιες. (Αν στη διαδρομή Β εκπαίδευσες για λίγο το βασικό μοντέλο,
# οι τιμές της απώλειας θα διαφέρουν από αυτές που δίνει το βιβλίο για την αποθηκευμένη κατάσταση εκπαίδευσης μετά από 3 εποχές.)
set_seed(42)  # ΠΡΙΝ αρχικοποιήσεις τον προσαρμογέα και ανακατέψεις τις παρτίδες (αν ορίσεις τον τυχαίο σπόρο μετά την εκπαίδευση, δεν αναπαράγεις τίποτα)
model = MiniGPT(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model = add_lora_to_model(model, rank=8, alpha=16)

# Μέτρησε τις παραμέτρους
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Σύνολο παραμέτρων:     {total:,}")
print(f"Εκπαιδεύσιμες παράμετροι (LoRA):     {trainable:,}")
print(f"Ποσοστό εκπαιδεύσιμων παραμέτρων: {100*trainable/total:.2f}%")

## 7. Βρόχος προσαρμογής με περαιτέρω εκπαίδευση

Εκπαίδευσε το μοντέλο ακολουθώντας τα 5 βήματα και χρησιμοποιώντας μάσκα απώλειας.

In [ ]:
import torch.nn.functional as F

def compute_masked_loss(logits, labels):
    """
    Υπολόγισε την απώλεια διασταυρούμενης εντροπίας, αγνοώντας τις θέσεις όπου labels == -100.

    Είναι σαν να βαθμολογείς μόνο την απάντηση σε ένα διαγώνισμα
    και όχι την ερώτηση που αντιγράφηκε από την προτροπή.
    """
    # Κάνε μετατόπιση για την πρόβλεψη του επόμενου token (όπως στο Κεφάλαιο 12)
    # Η .contiguous() εξασφαλίζει ότι τα δεδομένα βρίσκονται σε διαδοχικές θέσεις μνήμης για τη .view()
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()

    # Διασταυρούμενη εντροπία με ignore_index=-100
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100  # Το PyTorch αγνοεί αυτές τις θέσεις
    )

    return loss


def finetune_epoch(model, dataloader, optimizer, device):
    """
    Κάνε προσαρμογή με περαιτέρω εκπαίδευση για μία εποχή, ακολουθώντας τα 5 βήματα και χρησιμοποιώντας μάσκα απώλειας.
    """
    model.train()
    total_loss = 0

    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        # ===== ΤΑ 5 ΒΗΜΑΤΑ =====

        # Βήμα 1: Μηδένισε τις κλίσεις
        optimizer.zero_grad()

        # Βήμα 2: Κάνε ένα πρόσθιο πέρασμα
        logits = model(input_ids)

        # Βήμα 3: Υπολόγισε την απώλεια ΜΕ ΜΑΣΚΑ
        loss = compute_masked_loss(logits, labels)

        # Βήμα 4: Υπολόγισε τις κλίσεις με οπισθοδιάδοση
        loss.backward()

        # Βήμα 5: Ενημέρωσε τα βάρη
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

print("Οι συναρτήσεις εκπαίδευσης ορίστηκαν!")

In [ ]:
# ===== ΕΚΠΑΙΔΕΥΣΗ =====
num_epochs = 3
learning_rate = 1e-3  # Με τους προσαρμογείς LoRA μπορείς να χρησιμοποιήσεις υψηλότερο ρυθμό μάθησης απ’ ό,τι με την προσαρμογή όλων των βαρών με περαιτέρω εκπαίδευση

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=learning_rate  # Ίδιος βελτιστοποιητής όπως στο βιβλίο: AdamW με την προεπιλεγμένη τιμή του weight_decay
)

train_losses = []
print(f"Εκπαίδευση σε {len(train_data)} παραδείγματα για {num_epochs} εποχές\n")
for epoch in range(num_epochs):
    loss = finetune_epoch(model, train_loader, optimizer, device)
    train_losses.append(loss)
    print(f"Εποχή {epoch+1}: απώλεια με μάσκα {loss:.4f}")

print("\nΗ προσαρμογή με περαιτέρω εκπαίδευση ολοκληρώθηκε!")
print("Σημείωση: με το βασικό μας μοντέλο των 16 εκατ. παραμέτρων, που εκπαιδεύτηκε από το μηδέν σε κείμενα του Σαίξπηρ, περίμενε ασυνάρτητο κείμενο·")
print("το κέρδος είναι ότι βλέπεις την απώλεια να μειώνεται και μαθαίνεις πώς λειτουργεί ο προσαρμογέας. Για απαντήσεις με φυσική ροή,")
print("ξεκίνα με τα προεκπαιδευμένα βάρη του GPT-2 (με τον κώδικα φόρτωσης του Κεφαλαίου 11).")

## 8. Σχεδίασε το γράφημα της προόδου της εκπαίδευσης

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Εποχή', fontsize=12)
plt.ylabel('Απώλεια εκπαίδευσης', fontsize=12)
plt.title('Καμπύλη απώλειας κατά την προσαρμογή με περαιτέρω εκπαίδευση', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(train_losses) + 1))
plt.show()

print(f"\nΗ απώλεια μειώθηκε από {train_losses[0]:.4f} σε {train_losses[-1]:.4f}")
print(f"Βελτίωση: {(1 - train_losses[-1]/train_losses[0])*100:.1f}%")

## 9. Αξιολόγηση (ΜΕΤΑ την προσαρμογή με περαιτέρω εκπαίδευση)

Ας δούμε τι άλλαξε πραγματικά. Το βασικό μας μοντέλο των 16 εκατομμυρίων παραμέτρων εκπαιδεύτηκε από το μηδέν. Τα κείμενα που παράγει δεν βγάζουν νόημα ούτε πριν ΟΥΤΕ μετά την προσαρμογή (δεν έμαθε ποτέ να γράφει με ευχέρεια στα αγγλικά). Αυτό που μπορείς να μετρήσεις είναι η πτώση της απώλειας εκπαίδευσης. Μπορείς επίσης να δεις στην πράξη πώς λειτουργούν οι προσαρμογείς. Για να συγκρίνεις κείμενα που διαβάζονται φυσικά τόσο πριν όσο και μετά την προσαρμογή, χρειάζεσαι ένα προεκπαιδευμένο βασικό μοντέλο όπως το GPT-2 (δες τον κώδικα φόρτωσης στο Κεφάλαιο 11).

In [ ]:
def evaluate_model(model, test_examples, tokenizer, device):
    """
    Αξιολόγησε το μοντέλο μετά την προσαρμογή με περαιτέρω εκπαίδευση σε παραδείγματα δοκιμής.
    """
    model.eval()
    results = []

    for example in test_examples:
        instruction = example['instruction']
        expected = example['response']

        # Δημιούργησε κείμενο με την ΙΔΙΑ μορφή προθέματος που χρησιμοποιήθηκε στην εκπαίδευση
        prompt = build_prompt(instruction)
        input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

        # Απώλεια σε δεδομένα εκτός εκπαίδευσης: δώσε στο μοντέλο σε κάθε βήμα
        # το προηγούμενο token της ΣΩΣΤΗΣ απάντησης και αξιολόγησε μόνο τα token της απάντησης (ίδια
        # μάσκα απώλειας όπως στην εκπαίδευση, αλλά σε δεδομένα που οι προσαρμογείς δεν έχουν ξαναδεί)
        full_ids, full_labels = format_example(instruction, expected, tokenizer)
        with torch.no_grad():
            logits = model(full_ids.unsqueeze(0).to(device))
            resp_loss = compute_masked_loss(logits, full_labels.unsqueeze(0).to(device))

        output = model.generate(
            input_ids,
            max_new_tokens=50,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

        # Αποκωδικοποίησε μόνο τα νέα token, παραλείποντας το token τερματισμού EOS
        new_tokens = output[0][input_ids.shape[1]:].tolist()
        if tokenizer.eos_token_id in new_tokens:
            new_tokens = new_tokens[:new_tokens.index(tokenizer.eos_token_id)]
        generated = tokenizer.decode(new_tokens).strip()

        # Ακριβής αντιστοίχιση
        is_exact = generated.lower() == expected.lower()

        # Επικάλυψη λέξεων
        gen_words = set(generated.lower().split())
        exp_words = set(expected.lower().split())
        overlap = len(gen_words & exp_words) / max(len(exp_words), 1)

        results.append({
            'instruction': instruction,
            'expected': expected,
            'generated': generated,
            'exact_match': is_exact,
            'word_overlap': overlap,
            'response_loss': resp_loss.item()
        })

    # Σύνοψη
    accuracy = sum(r['exact_match'] for r in results) / len(results)
    avg_overlap = sum(r['word_overlap'] for r in results) / len(results)

    return {
        'exact_match_accuracy': accuracy,
        'average_word_overlap': avg_overlap,
        'avg_response_loss': sum(r['response_loss'] for r in results) / len(results),
        'detailed_results': results
    }

# Αξιολόγησε το μοντέλο στα δεδομένα που κράτησες εκτός εκπαίδευσης
print("Αξιολόγηση στο σύνολο δοκιμής...")
results = evaluate_model(model, eval_data, tokenizer, device)

print(f"\n===== ΑΠΟΤΕΛΕΣΜΑΤΑ ΑΞΙΟΛΟΓΗΣΗΣ =====")
print(f"Ακρίβεια ακριβούς αντιστοίχισης: {results['exact_match_accuracy']*100:.1f}%")
print(f"Μέση επικάλυψη λέξεων: {results['average_word_overlap']*100:.1f}%")
print("(Με ένα βασικό μοντέλο που εκπαιδεύτηκε από το μηδέν, είναι αναμενόμενο το ποσοστό να είναι κοντά στο 0%· δες την παραπάνω σημείωση.)")
print(f"Απώλεια απάντησης σε δεδομένα εκτός εκπαίδευσης: {results['avg_response_loss']:.2f}")


In [ ]:
# Εμφάνισε τα αναλυτικά αποτελέσματα
print("\n===== ΑΝΑΛΥΤΙΚΑ ΑΠΟΤΕΛΕΣΜΑΤΑ ΔΟΚΙΜΗΣ =====")
for r in results['detailed_results']:
    status = "✓" if r['exact_match'] else "○"
    print(f"\n{status} Ερ.: {r['instruction']}")
    print(f"  Αναμενόμενη απάντηση:  {r['expected']}")
    print(f"  Παραγόμενη απάντηση: {r['generated'][:100]}..." if len(r['generated']) > 100 else f"  Παραγόμενη απάντηση: {r['generated']}")
    print(f"  Επικάλυψη:   {r['word_overlap']*100:.0f}%")

In [ ]:
def show_before_after(base_model, finetuned_model, test_prompts, tokenizer, device):
    """
    Σύγκριση δίπλα δίπλα: το βασικό μοντέλο και το μοντέλο μετά την προσαρμογή με περαιτέρω εκπαίδευση.
    """
    print("=" * 70)
    print("ΠΡΙΝ και ΜΕΤΑ την προσαρμογή με περαιτέρω εκπαίδευση")
    print("=" * 70)

    for prompt in test_prompts:
        full_prompt = build_prompt(prompt)
        input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(device)

        base_output = base_model.generate(
            input_ids, max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id)
        ft_output = finetuned_model.generate(
            input_ids, max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id)

        base_text = tokenizer.decode(base_output[0][len(input_ids[0]):])
        ft_text = tokenizer.decode(ft_output[0][len(input_ids[0]):])

        print(f"\nΠροτροπή: {prompt}")
        print(f"  ΠΡΙΝ: {base_text.strip()}")
        print(f"  ΜΕΤΑ:  {ft_text.strip()}")
        print("-" * 70)

# Μοντέλο ΠΡΙΝ: ακριβές αντίγραφο των αρχικών βαρών (χωρίς προσαρμογή)
base_model_fresh = MiniGPT(config).to(device)
base_model_fresh.load_state_dict(base_model.state_dict())

test_prompts = ["What does TechStartup Inc do?", "How do I reset my password?"]
set_seed(42)  # Όρισε ξανά την αρχική τιμή της γεννήτριας τυχαίων αριθμών για να μπορείς να επαναλάβεις την παρακάτω σύγκριση με τα ίδια δείγματα.
show_before_after(base_model_fresh, model, test_prompts, tokenizer, device)

print("\nΚαι τα δύο κείμενα είναι ακαταλαβίστικα: το μοντέλο των 16 εκατομμυρίων παραμέτρων εκπαιδεύτηκε από το μηδέν, αλλά δεν έμαθε ποτέ να γράφει")
print("με ευχέρεια στα αγγλικά. Η απώλεια μειώθηκε μετρήσιμα και ο προσαρμογέας λειτουργεί όπως πρέπει.")
print("Φόρτωσε πρώτα τα βάρη του GPT-2 (Κεφάλαιο 11) για να συγκρίνεις κείμενα σε στρωτά αγγλικά πριν και μετά την προσαρμογή.")

## 10. Αποθήκευση και φόρτωση βαρών LoRA

In [ ]:
def save_lora_weights(model, filepath):
    """
    Αποθήκευσε μόνο τις παραμέτρους LoRA (πολύ μικρό αρχείο!).
    """
    lora_state_dict = {
        name: param for name, param in model.state_dict().items()
        if 'lora_' in name
    }
    torch.save(lora_state_dict, filepath)

    # Εμφάνισε το μέγεθος του αρχείου
    import os
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"Τα βάρη LoRA αποθηκεύτηκαν: {filepath} ({size_mb:.2f} MB)")



# Αποθήκευσε τα βάρη LoRA
save_lora_weights(model, 'techstartup_lora.pt')

# Σύγκρινε το μέγεθος του αρχείου με το μέγεθος ολόκληρου του μοντέλου
torch.save(model.state_dict(), 'full_model.pt')
full_size = os.path.getsize('full_model.pt') / (1024 * 1024)
lora_size = os.path.getsize('techstartup_lora.pt') / (1024 * 1024)
print(f"Το πλήρες μοντέλο αποθηκεύτηκε: full_model.pt ({full_size:.1f} MB)")
print(f"\nΤο LoRA είναι {full_size/lora_size:.0f} φορές μικρότερο από το πλήρες μοντέλο!")

In [ ]:
def load_lora_weights(model, filepath):
    """
    Φόρτωσε τα βάρη LoRA σε ένα μοντέλο που έχει ήδη επίπεδα LoRA.
    """
    lora_state_dict = torch.load(filepath, weights_only=True)
    model.load_state_dict(lora_state_dict, strict=False)
    print(f"Τα βάρη LoRA φορτώθηκαν από: {filepath}")



# Παράδειγμα φόρτωσης
print("\n===== Δοκιμή φόρτωσης και αποθήκευσης =====")
# Ο προσαρμογέας έχει νόημα μόνο με το ίδιο ακριβώς βασικό μοντέλο στο οποίο εκπαιδεύτηκε:
# Φόρτωσε ξανά ΠΡΩΤΑ το best_model.pt, μετά πρόσθεσε τη δομή LoRA και τέλος φόρτωσε τον προσαρμογέα.
new_model = MiniGPT(config).to(device)
new_model.load_state_dict(checkpoint["model_state_dict"])  # Πρώτα το βασικό μοντέλο!
new_model = add_lora_to_model(new_model, rank=8, alpha=16)

before_load = generate_response(new_model, "What does TechStartup Inc do?", tokenizer)
print(f"Πριν από τη φόρτωση του LoRA: {before_load[:60]}...")

load_lora_weights(new_model, 'techstartup_lora.pt')

after_load = generate_response(new_model, "What does TechStartup Inc do?", tokenizer)
print(f"Μετά τη φόρτωση του LoRA:  {after_load[:60]}...")
# Έλεγχος: το μοντέλο που φόρτωσες ξανά πρέπει να δίνει τα ίδια logits με το εκπαιδευμένο μοντέλο
probe = tokenizer.encode("[INST] What is SmartScheduler? [/INST]",
                         return_tensors="pt").to(device)
model.eval(); new_model.eval()
with torch.no_grad():
    torch.testing.assert_close(new_model(probe), model(probe), atol=1e-5, rtol=0)
print("Ο έλεγχος επιβεβαίωσε ότι τα logits είναι ίδια μετά την επαναφόρτωση")
print("\nΟι προσαρμογείς LoRA αποθηκεύτηκαν και φορτώθηκαν με επιτυχία!")


# ===== ΣΥΓΧΩΝΕΥΣΗ: ενσωμάτωσε το LoRA στο βασικό μοντέλο και ΑΦΑΙΡΕΣΕ τα περιβλήματα =====
def merge_lora_into_base(model):
    """
    Ενσωμάτωσε το LoRA στα βασικά βάρη και ΑΦΑΙΡΕΣΕ τα περιβλήματα του προσαρμογέα,
    ώστε το μοντέλο μετά τη συγχώνευση να είναι και πάλι ένα απλό MiniGPT (μόνιμη αλλαγή).
    """
    for block in model.blocks:
        lora = block.attn.qkv_proj
        if isinstance(lora, LoRALinear):
            # Ενσωμάτωσε τη διόρθωση στο βασικό βάρος: W + (A @ B * scale).T
            delta = lora.lora_A @ lora.lora_B * lora.scale
            lora.base.weight.data += delta.T  # Ανάστρεψε τον πίνακα για να ταιριάζει με τη σύμβαση της Linear
            # Αντικατάστησε το περίβλημα με το συγχωνευμένο επίπεδο χωρίς περίβλημα
            block.attn.qkv_proj = lora.base

    print("Τα βάρη LoRA συγχωνεύτηκαν στο βασικό μοντέλο")

# Κάνε τη συγχώνευση και έλεγξε ότι τα logits έμειναν ίδια και οι παράμετροι του προσαρμογέα αφαιρέθηκαν.
# ΣΗΜΕΙΩΣΗ: η συγχώνευση είναι μόνιμη. Τρέξε αυτό το κελί μόνο μετά τις παραπάνω συγκρίσεις.
probe = tokenizer.encode("[INST] What is SmartScheduler? [/INST]",
                         return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    adapter_logits = model(probe)

merge_lora_into_base(model)

with torch.no_grad():
    merged_logits = model(probe)
torch.testing.assert_close(merged_logits, adapter_logits, atol=1e-5, rtol=0)

total = sum(p.numel() for p in model.parameters())
print(f"Παράμετροι του συγχωνευμένου μοντέλου: {total:,} (οι προσαρμογείς αφαιρέθηκαν)")
assert not any(isinstance(m, LoRALinear) for m in model.modules())
print("Ο έλεγχος ισοδυναμίας μετά τη συγχώνευση πέτυχε")
# Γιατί αναστρέφουμε τον πίνακα; Η forward υπολογίζει το x @ A @ B, οπότε η μεταβολή έχει σχήμα (in, out)·
# η nn.Linear αποθηκεύει τα βάρη με σχήμα (out, in), γι’ αυτό χρησιμοποιούμε το delta.T. Η αντικατάσταση του περιβλήματος
# (και όχι απλώς ο μηδενισμός των προσαρμογέων) καταργεί τους δύο περιττούς πολλαπλασιασμούς πινάκων του προσαρμογέα.


## Σύνοψη

**Τι φτιάξαμε:**

1. **Σύνολο δεδομένων με ζεύγη οδηγίας/απάντησης** και μάσκα απώλειας
2. **Προσαρμογείς LoRA** με τους οποίους εκπαιδεύεται <1% των παραμέτρων
3. **Βρόχο προσαρμογής με περαιτέρω εκπαίδευση** που ακολουθεί τη διαδικασία των 5 βημάτων
4. **Αξιολόγηση** που συγκρίνει τη συμπεριφορά πριν και μετά (για να είμαστε ειλικρινείς: η απώλεια μειώνεται, αλλά όταν το βασικό μοντέλο ξεκινά από το μηδέν, το κείμενο παραμένει ακατανόητο)
5. **Αποθήκευση/φόρτωση προσαρμογέων** για χρήση στην πράξη

**Βασικά συμπεράσματα:**

- Η μάσκα απώλειας εστιάζει την εκπαίδευση σε αυτό που έχει σημασία (στις απαντήσεις, όχι στις οδηγίες)
- Το πόσες παραμέτρους εξοικονομείς εξαρτάται από τις διαστάσεις του επιπέδου και τον βαθμό. Μέτρησε τα εκπαιδεύσιμα βάρη στο δικό σου μοντέλο.
- Με παγωμένα βάρη στο βασικό μοντέλο περιορίζεται σημαντικά η **καταστροφική λήθη** (η απώλεια γνώσεων από την προεκπαίδευση)
- Για την προσαρμογή με περαιτέρω εκπαίδευση, η ποιότητα των δεδομένων έχει μεγαλύτερη σημασία από την ποσότητα
- Οι προσαρμογείς είναι πολύ μικροί και μπορείς να τους αλλάζεις ανάλογα με την εργασία

**Στη συνέχεια:** Στο Κεφάλαιο 14 θα δούμε πώς να σχεδιάζεις προτροπές για να παίρνεις καλύτερες απαντήσεις χωρίς να αλλάζεις καθόλου τα βάρη του μοντέλου!

## Προαιρετικό πείραμα: Παραλλαγή με τυχαία βάση (με σαφή σήμανση)

Στην κύρια διαδικασία παραπάνω χρησιμοποιούμε σε κάθε βήμα την αποθηκευμένη κατάσταση εκπαίδευσης του Κεφαλαίου 12. Σε αυτό το πείραμα επαναλαμβάνουμε τη διαδικασία LoRA σε μια **βάση με τυχαία αρχικά βάρη**, για να δεις γιατί έχει σημασία η αποθηκευμένη κατάσταση εκπαίδευσης. ΔΕΝ είναι η εκτέλεση του βιβλίου: εδώ η αρχική αξιολόγηση, η εκπαίδευση και η επαναφόρτωση χρησιμοποιούν τα ίδια τυχαία βάρη βάσης, χωριστά από τη διαδικασία που χρησιμοποιεί την αποθηκευμένη κατάσταση εκπαίδευσης. Περίμενε διαφορετική (συνήθως υψηλότερη) αρχική απώλεια και διαφορετικά παραγόμενα κείμενα.

In [ ]:
# ===== ΠΑΡΑΛΛΑΓΗ ΜΕ ΤΥΧΑΙΑ ΒΑΣΗ (πείραμα με σήμανση, ΟΧΙ η εκτέλεση του βιβλίου) =====
set_seed(42)
rand_base = MiniGPT(config).to(device)  # τυχαία αρχικοποίηση, ΧΩΡΙΣ αποθηκευμένη κατάσταση εκπαίδευσης
rand_model = MiniGPT(config).to(device)
rand_model.load_state_dict(rand_base.state_dict())  # ίδια τυχαία βάρη
rand_model = add_lora_to_model(rand_model, rank=8, alpha=16)

rand_optimizer = AdamW([p for p in rand_model.parameters() if p.requires_grad],
                       lr=1e-3, weight_decay=0.01)
rand_losses = []
for epoch in range(3):
    loss = finetune_epoch(rand_model, train_loader, rand_optimizer, device)
    rand_losses.append(loss)
    print(f"[τυχαία αρχικοποιημένη βάση] Εποχή {epoch+1}: απώλεια με μάσκα {loss:.4f}")

print(f"\nΑπώλειες με βάση την αποθηκευμένη κατάσταση εκπαίδευσης: {[f'{l:.4f}' for l in train_losses]}")
print(f"Απώλειες τυχαίας βάσης:     {[f'{l:.4f}' for l in rand_losses]}")
print("Ίδια διαδικασία, διαφορετική βάση: οι αριθμοί εξαρτώνται από τα αρχικά βάρη,")
print("γι’ αυτό ακριβώς πρέπει να φορτώσεις ξανά τον προσαρμογέα στη ΔΙΚΗ ΤΟΥ βάση.")

## Ασκήσεις

### Άσκηση 1: Διαφορετικοί βαθμοί LoRA

Δοκίμασε rank=4 και rank=16. Πώς επηρεάζει ο βαθμός την εκπαίδευση και τα αποτελέσματα;

In [ ]:
# Ο ΚΩΔΙΚΑΣ ΣΟΥ ΕΔΩ
# 1. Δημιούργησε ένα μοντέλο με rank=4
# 2. Εκπαίδευσέ το για 3 εποχές
# 3. Σύγκρινε την απώλεια και τα αποτελέσματα της αξιολόγησης με εκείνα για rank=8

### Άσκηση 2: Φτιάξε τη δική σου συλλογή συχνών ερωτήσεων και απαντήσεων

Δημιούργησε ένα σύνολο δεδομένων για ένα θέμα που γνωρίζεις (το σχολείο σου, το χόμπι σου κτλ.).

In [ ]:
# Ο ΚΩΔΙΚΑΣ ΣΟΥ ΕΔΩ
# 1. Δημιούργησε τουλάχιστον 30 ζεύγη οδηγίας/απάντησης
# 2. Κάνε προσαρμογή του μοντέλου με περαιτέρω εκπαίδευση
# 3. Δοκίμασέ το με δικές σου ερωτήσεις

### Άσκηση 3: Δοκιμή χωρίς μάσκα απώλειας

Τι συμβαίνει αν ΔΕΝ καλύψουμε τα token της οδηγίας με τη μάσκα απώλειας;

In [ ]:
# Ο ΚΩΔΙΚΑΣ ΣΟΥ ΕΔΩ
# 1. Τροποποίησε το σύνολο δεδομένων ώστε η μάσκα απώλειας να ΜΗΝ καλύπτει τα token της οδηγίας (όλες οι ετικέτες = τα πραγματικά token)
# 2. Εκπαίδευσε το μοντέλο
# 3. Σύγκρινε τα αποτελέσματα με εκείνα της έκδοσης που χρησιμοποιεί μάσκα απώλειας
# 4. Τι παρατηρείς;